In [1]:
from platform import python_version
print(python_version())

3.11.14


### Calculating DEGs statistics

### For each LFC/FDR cutoff set, we get a different set of DEGs
  - LFC: LFC cutoff and FDR_LFC cutoff
  - Pathway: fdr and pval pathway cutoff and min num of genes

### Up and Down DEGs simulation
  - Up and Down DEGs/DAPs
  - Up and Down in pathways

### there are 2 statistical tables
  - pval/fdr cutoff x degs
  - pval/fdr/geneset/quantile degs_in_pathway, num_pathways

In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import MTD
from libs.GDC_lib import GDC
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'
PSI_ID = 'TCGA-PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/config/all_lfc_cutoffs_TCGA-PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD
>>> Tumor
>>> case Tumor
	DEGs 19572
		Up (#9553)
		Dw (#10019)

Up-regulated per biotype
                               biotype     n
0                            IG_C_gene    13
1                      IG_C_pseudogene     2
2                            IG_J_gene    11
3                      IG_J_pseudogene     1
4                            IG_V_gene   122
5                      IG_V_pseudogene    42
6                              Mt_tRNA    13
7                                  TEC    96
8                            TR_C_gene     6
9                            TR_D_gene     1
10                           TR_J_gene     9
11                           TR_V_gene    43
12                     TR_V_pseudogene     2
13                              lncRNA  3139
14                               miRNA    77
15                            misc_RNA    15
16              polymorphic_pseudogene     4
17        

In [5]:
gdc = GDC(root0=ROOT0, root0_data=ROOT0_DATA)

gdc.memory_restriction = False

### Get all programs

In [6]:
force = False
verbose = False

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)

"; ".join(prog_list)

'TCGA; MATCH; TARGET; CGCI; CMI; APOLLO; BEATAML1.0; CPTAC; MP2PRT; ALCHEMIST; CCDI; CCG; CDDP_EAGLE; CTSP; EXCEPTIONAL_RESPONDERS; FM; HCMI; MMRF; NCICCR; OHSU; ORGANOID; RC; REBC; TRIO; VAREPOP; WCDT'

In [7]:
PROG_ID = 'ORGANOID'
PROG_ID = 'HCMI'
PROG_ID = 'CPTAC'
PROG_ID = 'TCGA'

df_psi = gdc.get_primary_sites(prog_id=PROG_ID, force=False, verbose=verbose)
print(len(df_psi))

if PROG_ID == 'TCGA':
    psi_id = 'TCGA-PAAD'
    df2 = df_psi[df_psi.psi_id == psi_id]
else:
    disease_id = 'PAAD'
    df2 = df_psi[df_psi.disease_id == disease_id]

if df2.empty:
    print("Error: No data found for the specified parameters.")
elif len(df2) == 1:
    row = df2.iloc[0]
    psi_id = row.psi_id
    primary_site = row.primary_site

    if PROG_ID == 'TCGA':
        print("Only one entry found for TCGA.")
        disease_id = None
        disease_context = None
        cbioportal_study_id = None
    else:
        disease_id = row.disease_id
        disease_context = row.disease_context
        cbioportal_study_id = row.cbioportal_study_id

    print("\t", psi_id, primary_site)
else:
    print("Multiple entries found for the specified parameters.")

    for _, row in df2.iterrows():
        psi_id = row.psi_id
        disease_id = row.disease_id
        primary_site = row.primary_site
        disease_context = row.disease_context
        cbioportal_study_id = row.cbioportal_study_id
        print("\t", psi_id, disease_id, primary_site, disease_context, cbioportal_study_id)


33
Only one entry found for TCGA.
	 TCGA-PAAD Pancreas


In [8]:
fname = gdc.fname_prim_site_cbio
filename = gdc.root0_data / fname

dfprogs = pdreadcsv(fname, gdc.root0_data)
print(len(dfprogs))
dfprogs = dfprogs[~pd.isnull(dfprogs.cbioportal_study_id)]
print(len(dfprogs))
prog_list = np.unique(dfprogs.prog_id)
prog_list

89
73


array(['CCLE', 'CPTAC', 'TARGET', 'TCGA'], dtype=object)

In [9]:
prog_id = 'CPTAC'
dfprogs[dfprogs.prog_id == prog_id].head(2)

,prog_id,gdc_project_id,disease_id,psi_id,primary_site,disease_context,cbioportal_study_id,mapping_status,recommended_mutation_source,notes,alternative_cbioportal_study_ids,review_notes
33,CPTAC,CPTAC-2,BRCA,brca_cptac_2020,Breast,Breast cancer context,brca_cptac_2020,reviewed_candidate_cptac,GDC MAF first; cBioPortal only if mapping is validated,CPTAC breast may not map to one simple public mutation study.,breast_cptac_gdc,cBioPortal has BRCA CPTAC 2020 and Breast CPTAC GDC 2025; validate which mat...
34,CPTAC,CPTAC-2,COAD,coad_cptac_2019,Colon / Rectum,Colon cancer / colorectal adenocarcinoma,coad_cptac_2019,curated_context_mapping,GDC MAF first; cBioPortal only if mapping is validated,Use only for colon/rectum context inside CPTAC-2.,NaN,NaN


### Is there Pancreas?

In [10]:
PROG_ID = 'CPTAC'
PSI_ID = 'pancreas_cptac_gdc'
DISEASE_ID = 'PAAD'

verbose=True

df_psi = gdc.get_primary_sites(prog_id=PROG_ID, verbose=verbose)
df_psi.head(3)

Table opened ((89, 12)) at '/home/flavio/uv/perturb_agent/data/gdc_to_cbioportal_study_mapping.tsv'


,prog_id,gdc_project_id,disease_id,psi_id,primary_site,disease_context,cbioportal_study_id,mapping_status,recommended_mutation_source,notes,alternative_cbioportal_study_ids,review_notes
0,CPTAC,CPTAC-2,BRCA,brca_cptac_2020,Breast,Breast cancer context,brca_cptac_2020,reviewed_candidate_cptac,GDC MAF first; cBioPortal only if mapping is validated,CPTAC breast may not map to one simple public mutation study.,breast_cptac_gdc,cBioPortal has BRCA CPTAC 2020 and Breast CPTAC GDC 2025; validate which mat...
1,CPTAC,CPTAC-2,COAD,coad_cptac_2019,Colon / Rectum,Colon cancer / colorectal adenocarcinoma,coad_cptac_2019,curated_context_mapping,GDC MAF first; cBioPortal only if mapping is validated,Use only for colon/rectum context inside CPTAC-2.,NaN,NaN
2,CPTAC,CPTAC-2,COAD,coad_cptac_2019,Colon/Rectum,Colon cancer / colorectal adenocarcinoma,coad_cptac_2019,added_from_first_mapping_table,cBioPortal if validated; GDC MAF remains primary for GDC-native mutation files,Use for CPTAC-2 colon/rectum context.,NaN,This row was present in the first mapping table but absent from the auxiliar...


In [11]:
PROG_ID = 'TCGA'
PSI_ID = 'TCGA-PAAD'
DISEASE_ID = 'PAAD'

verbose=True

df_psi = gdc.get_primary_sites(prog_id=PROG_ID, verbose=verbose)
df_psi.head(3)


Table opened ((33, 5)) at '/home/flavio/uv/perturb_agent/data/primary_site_program_TCGA.tsv'


,psi_id,primary_site,psi_id.1,disease_type,name
0,TCGA-ACC,Adrenal gland,TCGA-ACC,Adenomas and Adenocarcinomas,Adrenocortical Carcinoma
1,TCGA-PCPG,"Adrenal gland, Retroperitoneum and peritoneum, Other endocrine glands and re...",TCGA-PCPG,Paragangliomas and Glomus Tumors,Pheochromocytoma and Paraganglioma
2,TCGA-BLCA,Bladder,TCGA-BLCA,"Epithelial Neoplasms, NOS, Squamous Cell Neoplasms, Transitional Cell Papill...",Bladder Urothelial Carcinoma


In [12]:
prog_list = ['CPTAC', 'TARGET', 'TCGA']

In [13]:
verbose=False
force=False

imax_samples = 200

df_tumor, df_normal, df_gtex_ctrl = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

for prog_id in prog_list:
    print(">>>", prog_id)

    df_psi = gdc.get_primary_sites(prog_id=prog_id, force=False, verbose=verbose)

    for ipsi, row in df_psi.iterrows():

        psi_id = row.psi_id
        primary_site = row.primary_site

        gdc.set_primary_site(psi_id)

        print(f"{ipsi}) {psi_id} -{primary_site}", end=" - ")

        df_tumor, df_normal, df_gtex_ctrl = gdc.calc_file_expression_tumor_normal_gtex(
                    imax_tumor=100, imax_normal=50, force=force, verbose=verbose)
        
        if not df_tumor.empty:break
    break

print(len(df_tumor), len(df_normal), len(df_gtex_ctrl))

>>> CPTAC
Multiple entries found for the specified parameters.
0) brca_cptac_2020 -Breast - Warning: No samples found.
Insufficient expression data for brca_cptac_2020.
Multiple entries found for the specified parameters.
1) coad_cptac_2019 -Colon / Rectum - Warning: No samples found.
Insufficient expression data for coad_cptac_2019.
Multiple entries found for the specified parameters.
2) coad_cptac_2019 -Colon/Rectum - Warning: No samples found.
Insufficient expression data for coad_cptac_2019.
Multiple entries found for the specified parameters.
3) ovary_cptac_gdc -Ovary - Warning: No samples found.
Insufficient expression data for ovary_cptac_gdc.
Multiple entries found for the specified parameters.
4) ovary_cptac_gdc -Ovary / Peritoneum / female genital organs - Warning: No samples found.
Insufficient expression data for ovary_cptac_gdc.
5) brain_cptac_2020 -Brain - Error reading csv/tsv '/home/flavio/uv/perturb_agent/data/CPTAC/BRAIN/lfc/expression_gtex_for_brain_cptac_2020.tsv': 

In [23]:
print(df_tumor.shape)
df_tumor.head(3)

(60616, 102)


,geneid,symbol,tumor_1,tumor_2,tumor_3,tumor_4,tumor_5,tumor_6,tumor_7,tumor_8,...,tumor_91,tumor_92,tumor_93,tumor_94,tumor_95,tumor_96,tumor_97,tumor_98,tumor_99,tumor_100
0,ENSG00000000003,TSPAN6,1,5,2,15,4,1,1,1,...,4,5,21,13,6,2,1,0,9,2
1,ENSG00000000005,TNMD,7,47,6,35,62,9,4,36,...,32,71,56,0,4,1,3,14,35,21
2,ENSG00000000419,DPM1,11,13,27,20,1,6,26,3,...,8,12,77,0,58,14,6,17,33,21


In [22]:
print(df_normal.shape)
df_normal.head(3)

(60616, 52)


,geneid,symbol,normal_1,normal_2,normal_3,normal_4,normal_5,normal_6,normal_7,normal_8,...,normal_41,normal_42,normal_43,normal_44,normal_45,normal_46,normal_47,normal_48,normal_49,normal_50
0,ENSG00000000003,TSPAN6,0,0,0,1,1,2,7,0,...,0,0,0,1,0,1,1,4,1,2
1,ENSG00000000005,TNMD,4,1,7,9,26,14,15,10,...,13,26,19,5,30,10,7,17,5,9
2,ENSG00000000419,DPM1,3,1,1,10,8,4,18,1,...,4,5,8,8,19,3,6,5,4,5


In [16]:
print(len(df_gtex_ctrl))
df_gtex_ctrl.head(3)

0


""


In [17]:
method='deseq2'
verbose=True

df_lfc, msg = gdc.calc_lfc_table(psi_id=psi_id, 
                                 run_conda=True, 
                                 method=method, 
                                 imax_tumor=200, imax_normal=100, verbose=verbose)
                   
print(msg)
len(df_lfc)

Table opened ((60616, 102)) at '/home/flavio/uv/perturb_agent/data/CPTAC/BRAIN/lfc/expression_tumor_for_brain_cptac_2020.tsv'
Table opened ((60616, 52)) at '/home/flavio/uv/perturb_agent/data/CPTAC/BRAIN/lfc/expression_normal_for_brain_cptac_2020.tsv'
Error reading csv/tsv '/home/flavio/uv/perturb_agent/data/CPTAC/BRAIN/lfc/expression_gtex_for_brain_cptac_2020.tsv': No columns to parse from file
enough normal samples 49.


/home/flavio/uv/perturb_agent/notebooks/../src/libs/calc_degs_lib.py:59: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df2["_total_reads"] = df2[count_cols].sum(axis=1)


>>> LFC calc with deseq2 - running inside conda environment.
enough normal samples 49.
There are 99 tumor samples; enough normal samples 49.


50627

In [19]:
psi_id

'brain_cptac_2020'

In [18]:
df_lfc.head(30)

,ensembl_id,symbol,lfc,lfcSE,statistic,pval,fdr,baseMean,method
0,ENSG00000270722,RNVU1-31,24.116,1.095,22.025,1.664e-107,6.783e-103,15.058,DESeq2
1,ENSG00000210176,MT-TH,23.837,1.208,19.727,1.254e-86,2.556e-82,12.375,DESeq2
2,ENSG00000168026,TTC21A,-3.689,0.203,-18.135,1.677e-73,2.279e-69,83.861,DESeq2
3,ENSG00000197301,HMGA2-AS1,4.496,0.252,17.827,4.364e-71,4.447e-67,365.803,DESeq2
4,ENSG00000142185,TRPM2,4.064,0.230,17.649,1.037e-69,8.453e-66,38.397,DESeq2
5,ENSG00000277247,AC083809.1,-2.821,0.169,-16.729,8.030e-63,5.087e-59,365.840,DESeq2
6,ENSG00000226252,AL135960.1,-3.276,0.196,-16.718,9.684e-63,5.087e-59,81.510,DESeq2
7,ENSG00000262619,LINC00621,8.130,0.486,16.716,9.982e-63,5.087e-59,4932.046,DESeq2
8,ENSG00000241749,RPSAP52,5.403,0.337,16.025,8.591e-58,3.811e-54,29.774,DESeq2
9,ENSG00000169994,MYO7B,-3.389,0.212,-16.019,9.350e-58,3.811e-54,158.963,DESeq2


In [20]:
fname_lfc = gdc.fname_lfc % gdc.psi_id
fname_lfc

'lfc_brain_cptac_2020.tsv'

In [21]:
_ = pdwritecsv(df_lfc, fname_lfc, gdc.root_lfc)